# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: True
torch: 2.10.0+cu128 cuda available: True
GPU: NVIDIA A100-SXM4-80GB
name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 81148 MiB


In [2]:
!nvidia-smi

Mon Apr 27 03:35:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             56W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [3]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
DATASET_CSV = PROJECT_ROOT / "dataset/fitzpatrick17k_cleaned.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [ ]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [4]:
!git fetch origin main && git pull origin main

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.47 KiB | 0 bytes/s, done.
From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
   22e1ab7..64d51ef  main       -> origin/main
From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
Updating 22e1ab7..64d51ef
Fast-forward
 .DS_Store                          | Bin 10244 -> 10244 bytes
 .gitignore                         |   3 +-
 notebooks/colab_training.ipynb     |  70 ++++++++++++++++++++++++++-----------
 src/train_baseline_efficientnet.py |  14 +++++++-
 4 files changed, 65 insertions(+), 22 deletions(-)


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [5]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

We start at **64×64** for fast iteration. Once the recipe looks healthy, bump `IMAGE_SIZE` to 224 and re-run.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

In [9]:
IMAGE_SIZE   = 64
EPOCHS       = 1
BATCH_SIZE   = 128       # T4 / A100 handle this comfortably at 64x64. Drop to 32 at 224x224.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [12]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 64x64
Train batches: 100, Val batches: 25
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:01<00:00, 15.4MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.4567, 2.4956, 0.4562]
Epoch [1/1] train_loss=1.1150, train_acc=0.3647, val_loss=1.0910, val_acc=0.3811  ← best
Restored best weights from epoch 1 (val_loss=1.0910) for final evaluation.
Saved checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/checkpoint.pt
Saved metrics: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/metrics.json
Baseline test run complete.


## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [13]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

Latest checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/checkpoint.pt


In [14]:
!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/checkpoint.pt
Evaluating on 3191 samples (25 batches)
Evaluation Summary
Samples: 3191
Top-1 Accuracy: 0.3811
Macro AUROC:    0.5553
Macro AUPRC:    0.3711

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=602): acc=0.3571 macroAUROC=0.5571 macroAUPRC=0.3750
             benign: n=  101 AUROC=0.5256 AUPRC=0.1971
          malignant: n=   85 AUROC=0.6047 AUPRC=0.2089
     non-neoplastic: n=  416 AUROC=0.5409 AUPRC=0.7189
fitzpatrick_2 (n=920): acc=0.3587 macroAUROC=0.5642 macroAUPRC=0.3864
             benign: n=  112 AUROC=0.5192 AUPRC=0.1370
          malignant: n=  164 AUROC=0.6190 AUPRC=0.2792
     non-neoplastic: n=  644 AUROC=0.5543 AUPRC=0.7429
fitzpatrick_3 (n=665): acc=0.3805 macroAUROC=0.5441 macroAUPRC=0.3694
             benign: n=   91 AUROC=0.5011 AUPRC=0.1595
          malignant: 

In [15]:
!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

Wrote /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_043404_report.md


In [16]:
from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

Report: /content/drive/MyDrive/SkinLesionBiasReduction/logs/20260427_043404_report.md


# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260427_043404/checkpoint.pt` |
| Split | val |
| Image size | 64 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_cleaned.csv` |
| Image dir | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/images` |
| Generated at | 2026-04-27T04:39:51.277323 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 3191 |
| Top-1 Accuracy | 0.3811 |
| Macro AUROC | 0.5553 |
| Macro AUPRC | 0.3711 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 602 | 0.3571 | 0.5571 | 0.3750 |
| 2 | 920 | 0.3587 | 0.5642 | 0.3864 |
| 3 | 665 | 0.3805 | 0.5441 | 0.3694 |
| 4 | 554 | 0.4134 | 0.5340 | 0.3647 |
| 5 | 318 | 0.4119 | 0.5011 | 0.3424 |
| 6 | 132 | 0.4394 | 0.6384 | 0.4349 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.5256 | 85 / 0.6047 | 416 / 0.5409 |
| 2 | 112 / 0.5192 | 164 / 0.6190 | 644 / 0.5543 |
| 3 | 91 / 0.5011 | 100 / 0.6068 | 474 / 0.5245 |
| 4 | 75 / 0.5309 | 59 / 0.5498 | 420 / 0.5212 |
| 5 | 39 / 0.4383 | 29 / 0.5496 | 250 / 0.5155 |
| 6 | 8 / 0.6956 | 16 / 0.6185 | 108 / 0.6011 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 101 / 0.1971 | 85 / 0.2089 | 416 / 0.7189 |
| 2 | 112 / 0.1370 | 164 / 0.2792 | 644 / 0.7429 |
| 3 | 91 / 0.1595 | 100 / 0.2182 | 474 / 0.7305 |
| 4 | 75 / 0.1400 | 59 / 0.1779 | 420 / 0.7761 |
| 5 | 39 / 0.1051 | 29 / 0.1100 | 250 / 0.8120 |
| 6 | 8 / 0.2461 | 16 / 0.1751 | 108 / 0.8836 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.3811 |
| Balanced accuracy | 0.3895 |
| Macro F1 | 0.3257 |
| Weighted F1 | 0.4297 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.1424 | 0.3286 | 0.1987 | 426 |
| malignant | 0.1959 | 0.4658 | 0.2758 | 453 |
| non-neoplastic | 0.7648 | 0.3741 | 0.5025 | 2312 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 140 | 156 | 130 |
| malignant | 106 | 211 | 136 |
| non-neoplastic | 737 | 710 | 865 |


## 6. Scale up the image size (when ready)

Once the 64×64 numbers look directionally right (macro AUROC clearly above 0.63, malignant AUROC by FST less skewed), re-run section **4** with these overrides and let section **5** evaluate the new run:

```python
IMAGE_SIZE = 224
EPOCHS     = 40
BATCH_SIZE = 32   # may need 16 on a T4 to fit memory
```

Everything else stays the same.

## 7. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [ ]:
# Uncomment when you want to start cGAN training. Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [ ]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs